# OpenMythos R16 — 7B QLoRA Fine-Tuning (Colab)

Fine-tune Qwen2.5-Coder-7B on governance SFT data.

**Runtime:** GPU (T4 or better)  
**Time:** ~2-3 hours  
**Cost:** Free

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth transformers datasets trl accelerate bitsandbytes
!pip install -q huggingface_hub

In [ ]:
# Step 2: Download training data from HuggingFace or GitHub
import os

# Option A: Upload manually
# from google.colab import files
# uploaded = files.upload()

# Option B: Download from GitHub
!wget -q https://raw.githubusercontent.com/djimitflo/openmythos-benchmark/main/analysis/openmythos-apex-runs/datasets/r15-merged-sft.jsonl -O /content/r15-merged-sft.jsonl

# Verify
import json
count = 0
with open('/content/r15-merged-sft.jsonl') as f:
    for line in f:
        if line.strip():
            count += 1
print(f'Dataset: {count} examples')

In [ ]:
# Step 3: Load model with Unsloth
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-Coder-7B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_rslora=False,
    loftq_config=None,
)

print('Model loaded with LoRA!')

In [ ]:
# Step 4: Prepare dataset
from datasets import load_dataset

dataset = load_dataset("json", data_files="/content/r15-merged-sft.jsonl", split="train")

def format_example(ex):
    return f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['output']}"

dataset = dataset.map(lambda x: {"text": format_example(x)})
print(f'Dataset: {len(dataset)} examples')
print(f'Sample: {dataset[0]["text"][:200]}')

In [ ]:
# Step 5: Train
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LEN,
    args=TrainingArguments(
        output_dir="./output",
        num_train_epochs=5,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=20,
        logging_steps=10,
        save_strategy="steps",
        save_steps=50,
        fp16=True,
        optim="adamw_8bit",
        report_to="none",
    ),
)

print('Starting training...')
trainer.train()

In [ ]:
# Step 6: Save and convert to GGUF
model.save_pretrained("/content/openmythos-r16-7b")
tokenizer.save_pretrained("/content/openmythos-r16-7b")

# Merge LoRA and export
model = model.merge_and_unload()
model.save_pretrained("/content/openmythos-r16-7b-merged")
tokenizer.save_pretrained("/content/openmythos-r16-7b-merged")

print('Model saved!')

# Download
from google.colab import files
import shutil
shutil.make_archive('/content/openmythos-r16-7b', 'zip', '/content/openmythos-r16-7b')
files.download('/content/openmythos-r16-7b.zip')